# Day 67: Async Batch Inference Client

**Layer:** Production


## Concept Overview

An async batch client with asyncio and aiohttp sends N concurrent requests while respecting a concurrency limit, collecting TTFT and throughput metrics.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Async Batch Client

asyncio.Semaphore bounds concurrency; asyncio.gather runs all requests concurrently.


In [ ]:
import asyncio, time, numpy as np

async def single_inference(req_id, semaphore, prompt_len=256, output_len=100):
    async with semaphore:
        t0 = time.perf_counter()
        await asyncio.sleep(prompt_len * 0.0001 + 0.05)  # TTFT
        ttft = (time.perf_counter() - t0) * 1000
        await asyncio.sleep(output_len * 0.02)  # decode
        total = (time.perf_counter() - t0) * 1000
        return {"id":req_id,"ttft_ms":ttft,"total_ms":total,"tokens":output_len}

async def batch_client(n_requests=100, concurrency=10):
    sem = asyncio.Semaphore(concurrency)
    t0 = time.perf_counter()
    results = await asyncio.gather(*[single_inference(i, sem) for i in range(n_requests)])
    elapsed = time.perf_counter() - t0
    return results, elapsed

for conc in [1, 5, 10, 20, 50]:
    results, elapsed = asyncio.run(batch_client(100, conc))
    ttfts = [r["ttft_ms"] for r in results]
    print(f"concurrency={conc:>3}: {elapsed:.1f}s total | TTFT P50={np.percentile(ttfts,50):.0f}ms P99={np.percentile(ttfts,99):.0f}ms")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- An async batch client with asyncio and aiohttp sends N concurrent requests while respecting a concurrency limit, collecting TTFT and throughput metrics..
- An async batch client with asyncio and aiohttp sends N concurrent requests while....
- Day 67 implementation complete.
